<a href="https://colab.research.google.com/github/ck1972/University-GeoAI/blob/main/Mod2_Lab3g_GradCAM_XAI_Building_Segmentation_PreTrained_UNET_Part3_GitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**GradCAM-based Explainability for Building Footprint Segmentation**

## Install required libraries

In [ ]:
# Install segmentation_models_pytorch: for U-Net and other deep learning segmentation architectures
!pip install segmentation-models-pytorch --quiet

# Install albumentations: for efficient image augmentation and preprocessing
!pip install albumentations --quiet

# Install buildingregulariser: for refining and regularizing predicted building footprints
!pip install buildingregulariser --quiet

# Install rasterio: for reading and writing raster data such as satellite or aerial imagery
!pip install rasterio --quiet

## Import required libraries

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from rasterio import features
from shapely.geometry import shape
from rasterio.features import shapes
from buildingregulariser import regularize_geodataframe
from sklearn.metrics import f1_score, jaccard_score
from scipy.ndimage import binary_dilation, binary_erosion
import torch
from torch.utils.data import Dataset
from torch.utils.data import random_split, DataLoader
import segmentation_models_pytorch as smp
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F # for implementing GradCAM
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random

## Mount Drive and load inputs

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/')
os.chdir('/content/drive/MyDrive/Chit_TR9207')

Mounted at /content/drive/


## Load image and rasterize GeoJSON to create a mask

In [ ]:
# Load raster and geojson

# Load image
img_path = 'TR9207a.tif'
with rasterio.open(img_path) as src:
    image = src.read([1, 2, 3])  # Assume RGB bands
    transform = src.transform
    height, width = src.height, src.width
    crs = src.crs

# Load GeoJSON
gdf = gpd.read_file('Bld_DSG_TR9207.geojson')

# Rasterize buildings
mask = features.rasterize(
    ((geom, 1) for geom in gdf.geometry),
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype=np.uint8
)

## Create patches from image and mask

In [ ]:
# Create patacjes from image and mask
class PatchDataset(Dataset):
    def __init__(self, image, mask, patch_size=256):
        self.image = image
        self.mask = mask
        self.patch_size = patch_size
        self.coords = [
            (i, j)
            for i in range(0, image.shape[1] - patch_size + 1, patch_size)
            for j in range(0, image.shape[2] - patch_size + 1, patch_size)
        ]

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        i, j = self.coords[idx]
        img_patch = self.image[:, i:i+self.patch_size, j:j+self.patch_size]
        mask_patch = self.mask[i:i+self.patch_size, j:j+self.patch_size]
        return torch.tensor(img_patch / 255.0, dtype=torch.float32), torch.tensor(mask_patch, dtype=torch.long)

## Split data and create dataLoaders

In [ ]:
# Spit data and create dataloaders
dataset = PatchDataset(image, mask)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

## Load pretrained U-Net (transfer learning)

In [ ]:
# Load pretrained U-Net model
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
).to(device)

## Train the model (binary crossentropy)

In [ ]:
# Train the model
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(10):
    model.train()
    total_loss = 0
    correct, total = 0, 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device).float().unsqueeze(1)
        preds = model(imgs)
        loss = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds_binary = (torch.sigmoid(preds) > 0.5).float()
        correct += (preds_binary == masks).sum().item()
        total += torch.numel(masks)

    train_losses.append(total_loss / len(train_loader))
    train_accs.append(correct / total)

    # Validation
    model.eval()
    val_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device).float().unsqueeze(1)
            preds = model(imgs)
            val_loss += criterion(preds, masks).item()
            preds_binary = (torch.sigmoid(preds) > 0.5).float()
            correct += (preds_binary == masks).sum().item()
            total += torch.numel(masks)

    val_losses.append(val_loss / len(val_loader))
    val_accs.append(correct / total)

    print(f"Epoch {epoch+1}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, "
          f"Train Acc: {train_accs[-1]:.4f}, Val Acc: {val_accs[-1]:.4f}")

## Visualize training vs. validation loss and accuracy

To assess the model's learning progress, plot the training and validation loss and accuracy over epochs:

In [ ]:
# Plot Loss
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## Predict full image mask

In [ ]:
# Predict full image mask
model.eval()
pred_mask = np.zeros((height, width), dtype=np.uint8)

with torch.no_grad():
    for i in range(0, height - 256 + 1, 256):
        for j in range(0, width - 256 + 1, 256):
            patch = image[:, i:i+256, j:j+256]
            patch_tensor = torch.tensor(patch/255.0, dtype=torch.float32).unsqueeze(0).to(device)
            output = model(patch_tensor)
            output = torch.sigmoid(output)
            pred = (output.squeeze().cpu().numpy() > 0.5).astype(np.uint8)
            pred_mask[i:i+256, j:j+256] = pred

## Evaluate full-image segmentation with standard metrics

In [ ]:
# Evaluate full-image segmentation with standard metrics
from sklearn.metrics import f1_score, jaccard_score
from scipy.ndimage import binary_dilation, binary_erosion

# Flattened metrics
gt_flat = mask.flatten()
pred_flat = pred_mask.flatten()

# F1, IoU, Dice
f1 = f1_score(gt_flat, pred_flat)
iou = jaccard_score(gt_flat, pred_flat)
dice = 2 * (iou * f1) / (iou + f1)

# Compute Boundary Masks
def extract_boundary(mask, dilation_size=2):
    """Extract boundary of binary mask using morphological gradient (dilate - erode)."""
    dilated = binary_dilation(mask, iterations=dilation_size)
    eroded = binary_erosion(mask, iterations=dilation_size)
    boundary = np.logical_xor(dilated, eroded).astype(np.uint8)
    return boundary

gt_boundary = extract_boundary(mask)
pred_boundary = extract_boundary(pred_mask)

# Compute Boundary IoU
intersection = np.logical_and(gt_boundary, pred_boundary).sum()
union = np.logical_or(gt_boundary, pred_boundary).sum()
biou = intersection / union if union > 0 else 0

# Print metrics
print(f"F1 Score:         {f1:.4f}")
print(f"IoU:              {iou:.4f}")
print(f"Dice Coefficient: {dice:.4f}")
print(f"Boundary IoU:     {biou:.4f}")

## Display sample test patches

In [ ]:
# Select 3 random samples from the validation set
for _ in range(3):
    idx = random.randint(0, len(val_ds) - 1)
    img, gt = val_ds[idx]
    img_tensor = img.unsqueeze(0).to(device)

    # Predict the mask
    with torch.no_grad():
        pred = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()
    pred_binary = (pred > 0.5).astype(np.uint8)

    # Plot the results
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img.permute(1, 2, 0))
    plt.title("Input Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(gt.numpy(), cmap='gray')
    plt.title("Ground Truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(pred_binary, cmap='gray')
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

## Convert mask to polygons and save GeoJSON

In [ ]:
# Convert building masks to polygons
results = (
    {'properties': {'value': v}, 'geometry': s}
    for s, v in shapes(pred_mask, transform=transform)
    if v == 1
)

gdf_pred = gpd.GeoDataFrame.from_features(results, crs=crs)
gdf_pred.to_file("pred_pretrained_UNET_bld9207.geojson", driver="GeoJSON")

## Regularize predicted building polygons

In [ ]:
# Vectorize the predicted mask
results = (
    {"properties": {"raster_val": v}, "geometry": s}
    for s, v in shapes(pred_mask, mask=pred_mask.astype(bool), transform=transform)
)

# Create a GeoDataFrame
gdf_pred = gpd.GeoDataFrame.from_features(results, crs=crs)

# Regularize the building footprints
gdf_reg = regularize_geodataframe(gdf_pred)

# Save the regularized footprints to a GeoJSON file
gdf_reg.to_file("pred_pretrained_UNET_bld9207_regularized.geojson", driver="GeoJSON")

## Save the trained model (weights only)

In [ ]:
# Save model weights
model_path = "pretrained_unet_building_segmentation.pth"  # ✅ clear and descriptive name
torch.save(model.state_dict(), model_path)                # ✅ saves only the weights
print(f"Model weights saved to: {model_path}")            # ✅ confirmation output

## Implementing GradCAM for semantic segmentation
### Define GradCAM class

In [ ]:
# Define GradCAM class for segmentation models like U-Net
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval()  # Set model to evaluation mode
        self.target_layer = target_layer  # Layer to visualize
        self.activations = None  # Store feature maps (forward pass)
        self.gradients = None    # Store gradients (backward pass)
        self._register_hooks()   # Register forward and backward hooks

    def _register_hooks(self):
        # Save activations from forward pass
        def forward_hook(module, input, output):
            self.activations = output

        # Save gradients from backward pass
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        # Attach hooks to the target layer
        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def __call__(self, input_tensor):
        input_tensor.requires_grad = True  # Enable gradient tracking
        output = self.model(input_tensor)  # Forward pass
        pred = torch.sigmoid(output)       # Apply sigmoid activation
        score = pred.mean()                # Use mean output for backward pass
        self.model.zero_grad()             # Zero previous gradients
        score.backward(retain_graph=True)  # Backward pass to compute gradients

        # Compute channel-wise importance
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])

        # Get activations for the input image
        activations = self.activations[0]
        for i in range(len(pooled_gradients)):
            activations[i] *= pooled_gradients[i]

        # Average over channels to get the heatmap
        heatmap = torch.mean(activations, dim=0).cpu().detach().numpy()

        # Normalize heatmap to [0, 1]
        heatmap = np.maximum(heatmap, 0)
        heatmap /= np.max(heatmap) if np.max(heatmap) > 0 else 1
        return heatmap

# Select the last decoder convolutional layer of the U-Net model
target_layer = model.decoder.blocks[-1].conv1

# Instantiate the GradCAM explainer
gradcam = GradCAM(model, target_layer)

### Visualizing GradCAM overlays on predicted masks

In [ ]:
for _ in range(3):
    # Randomly select a validation sample
    idx = random.randint(0, len(val_ds) - 1)
    img, gt_mask = val_ds[idx]  # Load image and ground truth mask
    input_tensor = img.unsqueeze(0).to(device)  # Add batch dim and move to GPU/CPU

    # Predict the segmentation mask
    with torch.no_grad():
        pred_logits = model(input_tensor)
        pred_mask = torch.sigmoid(pred_logits).squeeze().cpu().numpy()
    pred_binary = (pred_mask > 0.5).astype(np.uint8)  # Binarize mask at 0.5

    # Generate GradCAM heatmap
    cam = gradcam(input_tensor)

    # Normalize and convert heatmap to RGB color map
    cam_uint8 = np.uint8(255 * cam)  # Scale to 0–255
    cam_color = cv2.applyColorMap(cam_uint8, cv2.COLORMAP_JET)
    cam_color = cv2.cvtColor(cam_color, cv2.COLOR_BGR2RGB)

    # Resize heatmap to match input image
    img_np = img.permute(1, 2, 0).numpy()  # Convert to HWC format
    h, w = img_np.shape[:2]
    cam_resized = cv2.resize(cam_color, (w, h))

    # Overlay heatmap on original image
    overlay_on_image = cv2.addWeighted((img_np * 255).astype(np.uint8), 0.6, cam_resized, 0.4, 0)

    # Convert binary mask to RGB and overlay GradCAM
    pred_mask_rgb = np.stack([pred_binary * 255] * 3, axis=-1).astype(np.uint8)
    overlay_on_pred = cv2.addWeighted(pred_mask_rgb, 0.6, cam_resized, 0.4, 0)

    # Plot the visualizations
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(img_np)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(gt_mask.numpy(), cmap="gray")
    plt.title("Ground Truth Mask")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(pred_binary, cmap="gray")
    plt.title("Predicted Mask")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay_on_pred)
    plt.title("GradCAM on Predicted Mask")
    plt.axis("off")

    plt.tight_layout()
    plt.show()